# CfC 液态神经网络 PYNQ 上板演示（LNN CfC Demo V1.0）

本 Notebook 按方案 §7.1 的六步现场流程演示 **1 输入 / 4 隐状态 / 1 输出** 的 CfC 单步计算图在 PL 上的连续 64 步推理：

1. 加载匹配的 overlay（`lnn_cfc.bit` + `lnn_cfc.hwh`，同目录同 basename），显示 CFC1 ID、版本、时钟频率与部署文件哈希；
2. 选择固定测试样本，显示 64 个输入点、ΔT 与初始状态，执行浮点与定点参考；
3. 一次写入输入缓冲与配置，启动 FPGA，等待 DONE/ERROR（独立墙钟超时），读回全部输出与末状态；
4. **先做整数逐点比对**，通过后再叠加 输入 / 目标 / 浮点参考 / FPGA 输出 四条曲线，显示任务 MSE、周期数、硬件耗时与端到端耗时；
5. 切换"每步清状态"模式，运行同一输入并比较任务误差，解释状态记忆的作用；
6. 恢复正常模式，确认状态清零与重复运行可复现，并保存本次结果。

> **数据来源标注（必须保留）**
> - 步骤 1 的横幅会明确显示 `source = hardware`（真实板卡本次实测）或 `source = simulation-model`
>   （离线 Python 寄存器模型，**未在真实板卡上运行，不能作为板端实测证据**）。
> - 每个指标表下方都会再次显示该标注；离线结果不得用于验收报告。
>
> 运行环境：板卡上需 PYNQ 镜像（`pynq` 包）、matplotlib、numpy；把整个交付包（含 `export/`）拷到板子同一目录，
> 用 Jupyter 打开本 Notebook。开发机上无板时会自动落到离线模型，仅用于流程自检。

> **部署（必须让本 Notebook 与 `export/` 同目录）**
> - 推荐：把整个 `board/` 目录拷到板子（已自带 `board/export/` 期望值），在本目录打开本 Notebook；
> - 或把正式包 `board/package/` 拷到板子（内含 `export/`）；
> - 若已拷好驱动/位流但缺 `export/`：scp -r <主机>/board/export xilinx@<板IP>:<本目录>/export，
>   或重启内核前设 os.environ["CFC_EXPECT_DIR"] = "/path/to/export"。


## 0 · 环境检查与路径解析


In [ ]:
# ============================================================
# 0) 环境与路径（无板时自动使用离线寄存器模型，并全程如实标注）
# ============================================================
import json
import os
import sys
import time
from pathlib import Path

HERE = Path.cwd()
for cand in (HERE, HERE / "board", HERE.parent / "board", HERE.parent):
    if (cand / "lnn_cfc_run.py").is_file():
        sys.path.insert(0, str(cand))
        break
import lnn_cfc_run as drv
import cfc_board_ui as ui

# 现场可按需修改（也可用环境变量覆盖）
BIT = os.environ.get("CFC_BIT", "")            # 例如 "lnn_cfc.bit"
XSA = os.environ.get("CFC_XSA", "")
EXPECT_DIR = os.environ.get("CFC_EXPECT_DIR", "")
SAMPLE_ID = os.environ.get("CFC_SAMPLE", "sample_000")
BACKEND = os.environ.get("CFC_BACKEND", "auto")  # auto | hardware | simulation-model
TIMEOUT_S = float(os.environ.get("CFC_TIMEOUT_S", "10"))
ALLOW_FIXTURE = os.environ.get("CFC_ALLOW_FIXTURE", "0") == "1"  # 仅限合成 fixture 流程自检
RESULTS_DIR = Path(os.environ.get("CFC_RESULTS_DIR", str(HERE / "results")))

PATHS = drv.discover_paths(bit=BIT or None, xsa=XSA or None,
                           expect_dir=EXPECT_DIR or None)
ui.show_environment(dict(PATHS, results_dir=str(RESULTS_DIR)))
print("driver  :", drv.__file__)
print("backend :", BACKEND, "| sample:", SAMPLE_ID, "| timeout_s:", TIMEOUT_S)


## 1 · 加载 overlay，核对 CFC1 ID / 版本 / 时钟 / 哈希


In [ ]:
# ---------- 步骤 1：加载 overlay，显示 CFC1 / 版本 / 时钟 / 部署文件哈希 ----------
SESSION = None
REPORT = None
try:
    SESSION = drv.BoardSession(bit=BIT or None, xsa=XSA or None,
                               expect_dir=EXPECT_DIR or None,
                               backend=BACKEND, verbose=False,
                               allow_fixture=ALLOW_FIXTURE)
    REPORT = SESSION.bringup()
    ui.show_bringup(REPORT)
except Exception as exc:
    ui.show_error(exc)
    SESSION = None
    REPORT = None


## 2 · 选择固定测试样本，展示输入 / ΔT / 初始状态，执行浮点与整数参考


In [ ]:
# ---------- 步骤 2：固定测试样本（64 输入点 / ΔT / 初始状态 / 浮点与整数参考） ----------
SAMPLE = None
if SESSION is not None:
    try:
        SAMPLE = SESSION.load_sample(SAMPLE_ID)
        ui.show_sample(SAMPLE, save_path=RESULTS_DIR / "fig_sample.png")
    except Exception as exc:
        ui.show_error(exc)
else:
    print("bring-up 未成功，先修复步骤 1。")


## 3 · 写输入与配置，启动 FPGA，等待 DONE/ERROR，读回输出与末状态


In [ ]:
# ---------- 步骤 3：一次写入输入与配置，启动 FPGA，等待 DONE/ERROR，读回结果 ----------
RESULT = None
if SAMPLE is not None:
    try:
        RESULT = SESSION.run(SAMPLE, mode="normal", timeout_s=TIMEOUT_S)
        ui.show_run(RESULT)
    except Exception as exc:
        ui.show_error(exc)
else:
    print("样本未加载，跳过。")


## 4 · 先整数逐点比对，再画 输入 / 目标 / 浮点参考 / FPGA 输出 四条曲线


In [ ]:
# ---------- 步骤 4：整数逐点比对通过后，画四条曲线并显示实测指标 ----------
if RESULT is not None and RESULT.get("status") == "done":
    ui.show_compare(RESULT)
    if (RESULT.get("compare") or {}).get("outputs_bit_exact"):
        ui.plot_four_curves(SAMPLE, RESULT,
                            save_path=RESULTS_DIR / "fig_four_curves.png")
        ui.show_metrics(RESULT)
    else:
        print("整数比对未通过：先排查（不画成功曲线、不展示为 PASS）。")
        ui.show_metrics(RESULT)
elif RESULT is not None:
    print("运行未成功（status=%s），错误与诊断见步骤 3。" % RESULT.get("status"))
    ui.show_metrics(RESULT)


## 5 · 切换每步清状态模式，对比任务误差（状态记忆）


In [ ]:
# ---------- 步骤 5：每步清状态模式对比（状态记忆的作用） ----------
RESULT_CLEAR = None
if SAMPLE is not None and RESULT is not None:
    try:
        RESULT_CLEAR = SESSION.run(SAMPLE, mode="clear-each-step", timeout_s=TIMEOUT_S)
        ui.show_run(RESULT_CLEAR)
        ui.show_mode_compare(RESULT, RESULT_CLEAR, SAMPLE,
                             save_path=RESULTS_DIR / "fig_mode_compare.png")
    except Exception as exc:
        ui.show_error(exc)
else:
    print("需要先完成步骤 3 的正常模式运行。")


## 6 · 恢复正常模式，确认清零重复运行与 32+32 分段一致


In [ ]:
# ---------- 步骤 6：正常模式重复运行 + 32+32 分段保留状态，确认可复现 ----------
CHECKS = None
if SAMPLE is not None:
    try:
        CHECKS = SESSION.run_checks(SAMPLE, timeout_s=TIMEOUT_S)
        ui.show_repeat_and_segments(CHECKS)
    except Exception as exc:
        ui.show_error(exc)
else:
    print("样本未加载，跳过。")


## 7 · 保存本次结果


In [ ]:
# ---------- 保存本次结果（summary.json / outputs.json / outputs.npy / 曲线 PNG） ----------
SAVED = {}
if SESSION is not None and RESULT is not None:
    try:
        stamp = time.strftime("%Y%m%d_%H%M%S", time.localtime())
        tag = "%s_%s_%s" % (RESULT.get("sample_id"), RESULT.get("mode"), SESSION.kind)
        out = RESULTS_DIR / (stamp + "_" + tag)
        SAVED = SESSION.save_result(RESULT, out_dir=out)
        if CHECKS is not None:
            SAVED["state_checks"] = SESSION.save_checks(CHECKS, out_dir=out)
        if RESULT_CLEAR is not None:
            saved_clear = SESSION.save_result(RESULT_CLEAR,
                                              out_dir=out / "clear_each_step")
            SAVED["clear_each_step_dir"] = saved_clear["dir"]
        ui.show_saved(SAVED)
        if RESULT.get("source") == "hardware":
            print("本次结果标注：真实板卡本次实测（可写入验收报告）。")
        else:
            print("本次结果标注：离线寄存器模型 simulation-model，未在真实板卡上运行，"
                  "不得作为板端实测证据。")
    except Exception as exc:
        ui.show_error(exc)
else:
    print("没有可保存的运行结果。")


## 结果判读与已知限制

- **验收口径**：FPGA 全部 64 个输出与 export 定点 golden 逐位一致、末状态一致、`STEP_COUNT == N_STEPS`、
  `ID/VERSION/CLOCK_HZ` 正确，才记为 PASS。浮点曲线接近不能代替逐点整数比对。
- **两类误差**：任务 MSE（FPGA 输出 vs 目标 `x[t-4]`，跳过前 5 步预热）反映模型效果；
  定点误差（FPGA 输出 vs 定点 golden）反映硬件正确性，本流程要求为 0。
- **性能读数**：`硬件耗时 = RUN_CYCLES / CLOCK_HZ`，只含片上计算（START 被接受到最后输出提交）；
  端到端耗时含 PS 写入、轮询等待与读回，由板端单调时钟独立测量，两者要分开引用。
- **循环播放/重复演示**：帧率与 300 MHz 真实硬件计算时间是两回事，报告里不要混用。
- **离线运行说明**：若步骤 1 显示 `simulation-model`，说明当前没有可用板卡/overlay，
  所有数字来自 Python 寄存器模型，仅用于流程自检。
- 设备报错时每个步骤会直接显示错误、ERROR_CODE 语义与寄存器/overlay 诊断，不会静默跳成 PASS。
